In [44]:
using LowLevelFEM
import LowLevelFEM as L
using LinearAlgebra

In [45]:
structured_rect_mesh(n=200)
mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=2, field=:u)

Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 40401, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs)

In [46]:
#export ChainSum, TransposedChain, withgauss, reduced, full

import Base: +, adjoint, transpose
import LowLevelFEM: ⋅

In [47]:
"""
    ChainSumTerm(chain, gauss)

Internal storage for one term of a compound operator expression.

`chain` is usually an `OpApplied` or a `MatrixChain`.
`gauss` stores the preferred integration rule for this term.
"""
struct ChainSumTerm
    chain::Any
    gauss::Any
end


ChainSumTerm

In [48]:
"""
    ChainSum(terms)

Symbolic sum of operator chains.

Example:

```julia
B = Grad(Pu) ⋅ A + Pu ⋅ G
```

stores two terms without assembling anything.
"""
struct ChainSum
    terms::Vector{ChainSumTerm}
end


ChainSum

In [49]:
"""
TransposedChain(chain)

Symbolic transpose marker for compound expressions.

This does not assemble or numerically transpose anything immediately.
It is interpreted during expansion of the bilinear form.
"""
struct TransposedChain
    chain::Any
end


TransposedChain

In [50]:
"""
CompoundChain(left, mats)

Temporary object representing

```
left ⋅ M1 ⋅ M2 ⋯
```

before the right-hand side is supplied.
"""
struct CompoundChain
    left::Any
    mats::Vector{Any}
end


CompoundChain

In [51]:
"""
CompoundBilinear(left, mats, right)

Temporary object representing

```
left ⋅ M1 ⋅ M2 ⋯ ⋅ right
```

where `left` and/or `right` may be `ChainSum`.
"""
struct CompoundBilinear
    left::Any
    mats::Vector{Any}
    right::Any
end


CompoundBilinear

In [52]:
withgauss(chain, gauss) = ChainSumTerm(chain, gauss)
reduced(chain) = withgauss(chain, :reduced)
full(chain) = withgauss(chain, :full)

_as_sumterm(t::ChainSumTerm) = t
_as_sumterm(x) = ChainSumTerm(x, :full)

_chain_sum(xs...) = ChainSum([_as_sumterm(x) for x in xs])

_is_chain_object(x) =
    x isa L.OpApplied ||
    x isa L.MatrixChain ||
    x isa ChainSum ||
    x isa ChainSumTerm ||
    x isa TransposedChain

function _coeff_mats(c)
    if c isa AbstractMatrix
        _check_coeff_matrix(c)
        return Any[c]
    elseif c isa TensorField
        return Any[tensorfield_to_matrix(c)]
    elseif c isa Number || c isa ScalarField
        return Any[c]
    else
        error("Compound operator: invalid coefficient type $(typeof(c)).")
    end
end

function _op_and_mats(x)
    if x isa L.OpApplied
        return x, Any[]
    elseif x isa L.MatrixChain
        return x.a, copy(x.mats)
    else
        error("Compound operator term must be OpApplied or MatrixChain, got $(typeof(x)).")
    end
end

struct _SideTerm
    chain::Any
    transposed::Bool
    gauss::Any
end

function _side_terms(x; transposed=false)
    if x isa TransposedChain
        return _side_terms(x.chain; transposed=(!transposed))
    elseif x isa ChainSum
        return [_SideTerm(t.chain, transposed, t.gauss) for t in x.terms]

    elseif x isa ChainSumTerm
        return [_SideTerm(x.chain, transposed, x.gauss)]

    elseif x isa L.OpApplied || x isa L.MatrixChain
        return [_SideTerm(x, transposed, :full)]

    else
        error("Compound operator: invalid side type $(typeof(x)).")
    end
end

function _maybe_transpose_mats(mats, transposed::Bool)
    if transposed
        return Any[adjoint(M) for M in reverse(mats)]
    else
        return copy(mats)
    end
end

function _select_gauss(left::_SideTerm, right::_SideTerm, gauss)
    gauss !== :auto && return gauss
    if left.gauss == right.gauss
        return left.gauss
    end

    return :full
end

function _compress_mats(mats)
    isempty(mats) && return 1.0

    C = mats[1]
    for i in 2:length(mats)
        C = C * mats[i]
    end

    return C
end

function _make_bilinear_term(left::_SideTerm, middle::Vector{Any}, right::_SideTerm)
    left_op, left_mats = _op_and_mats(left.chain)
    right_op, right_mats = _op_and_mats(right.chain)
    mats = Any[
        #_maybe_transpose_mats(left_mats, left.transposed)...,
        left_mats...,
        middle...,
        #_maybe_transpose_mats(right_mats, right.transposed)...
        adjoint.(reverse(right_mats))...
    ]

    coef = isempty(mats) ? 1.0 : mats

    #coef = _compress_mats(mats)

    return L.BilinearTerm(left_op, coef, right_op)
end

function _coeff_mats(c)
    if c isa AbstractMatrix
        return Any[c]
    elseif c isa Number || c isa ScalarField || c isa TensorField
        return Any[c]
    else
        error("Compound operator: invalid coefficient type $(typeof(c)).")
    end
end

_coeff_mats (generic function with 1 method)

In [53]:
# ---------------------------------------------------------------------------

# Addition: build ChainSum

# ---------------------------------------------------------------------------

+(a::ChainSum, b::ChainSum) =
    ChainSum([a.terms...; b.terms...])

+(a::ChainSum, b::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain}) =
    ChainSum([a.terms...; _as_sumterm(b)])

+(a::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain}, b::ChainSum) =
    ChainSum([_as_sumterm(a); b.terms...])

+(a::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain},
    b::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain}) =
    _chain_sum(a, b)

+ (generic function with 247 methods)

In [54]:
# ---------------------------------------------------------------------------

# Symbolic transpose

# ---------------------------------------------------------------------------

adjoint(x::Union{ChainSum,ChainSumTerm,L.MatrixChain,L.OpApplied}) =
    TransposedChain(x)

transpose(x::Union{ChainSum,ChainSumTerm,L.MatrixChain,L.OpApplied}) =
    TransposedChain(x)

transpose (generic function with 53 methods)

In [55]:
# ---------------------------------------------------------------------------

# Dot products involving ChainSum / TransposedChain

# ---------------------------------------------------------------------------

function ⋅(left::Union{ChainSum,TransposedChain,ChainSumTerm}, c)
    return CompoundChain(left, _coeff_mats(c))
end

function ⋅(cc::CompoundChain, c)
    append!(cc.mats, _coeff_mats(c))
    return cc
end

function ⋅(cc::CompoundChain, right::Union{ChainSum,TransposedChain,ChainSumTerm,L.OpApplied,L.MatrixChain})
    return CompoundBilinear(cc.left, cc.mats, right)
end

function ⋅(left::Union{ChainSum,TransposedChain,ChainSumTerm},
    right::Union{ChainSum,TransposedChain,ChainSumTerm,L.OpApplied,L.MatrixChain})
    return CompoundBilinear(left, Any[], right)
end

function ⋅(left::Union{L.OpApplied,L.MatrixChain},
    right::Union{ChainSum,TransposedChain,ChainSumTerm})
    return CompoundBilinear(left, Any[], right)
end

dot (generic function with 74 methods)

In [56]:
function ⋅(A::AbstractMatrix, op::L.OpApplied)
    return op ⋅ adjoint(A)
end

function ⋅(A::AbstractMatrix, ch::L.MatrixChain)
    return ch ⋅ adjoint(A)
end

function ⋅(A::Union{Number,ScalarField}, op::L.OpApplied)
    return op ⋅ A
end

dot (generic function with 74 methods)

In [57]:
# ---------------------------------------------------------------------------

# Integral wrapper

# ---------------------------------------------------------------------------

function ∫(expr::CompoundBilinear;
    Ω=nothing, Γ=nothing, weight=nothing,
    gauss=:auto)
    #@show typeof(expr.left)
    #@show typeof(expr.right)

    left_terms = _side_terms(expr.left)
    right_terms = _side_terms(expr.right)
    #@show left_terms
    #@show right_terms

    K = nothing

    for LL in left_terms
        for R in right_terms
            bt = _make_bilinear_term(LL, expr.mats, R)
            #@show bt
            g = _select_gauss(LL, R, gauss)

            Ki = L.∫(bt; Ω=Ω, Γ=Γ, weight=weight, gauss=g)
            K = K === nothing ? Ki : K + Ki
        end
    end

    return K
end

∫ (generic function with 1 method)

In [58]:
A = [1 0 0 0; 0 1 0 0]
G = [1 0; 0 1]
M = [1 0; 0 1]

2×2 Matrix{Int64}:
 1  0
 0  1

In [59]:
B = Grad(Pu) ⋅ A + Pu ⋅ G

ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 40401, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.GradOp()), Any[[1 0 0 0; 0 1 0 0]]), :full), ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 40401, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.IdOp()), Any[[1 0; 0 1]]), :full)])

In [60]:
B' ⋅ M ⋅ B

CompoundBilinear(TransposedChain(ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 40401, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.GradOp()), Any[[1 0 0 0; 0 1 0 0]]), :full), ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 40401, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.IdOp()), Any[[1 0; 0 1]]), :full)])), Any[[1 0; 0 1]], ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_re

In [61]:
B = full(A ⋅ Grad(Pu)) + reduced(G ⋅ Pu)

K = ∫(B' ⋅ M ⋅ B; Ω="body", gauss=:auto)

sparse([1, 2, 9, 10, 1599, 1600, 1601, 1602, 1, 2  …  807, 808, 80401, 80402, 80403, 80404, 80799, 80800, 80801, 80802], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802], [0.6650035440710214, -0.0008333333333333333, -0.16666501119203062, -0.0004166666666666663, -0.16749937740435455, -0.0008333333333333336, -0.3333329054746363, -0.00041666666666666664, -0.0008333333333333333, 3.5440710214361918e-6  …  0.0004166666666666745, 4.278586968845227e-7, -0.00041666666666667065, 4.278586968845099e-7, 2.0057740190981832e-18, 3.310949272896065e-6, -0.0016666666666666672, 2.4480043512778716e-6, -1.214306433183765e-17, 1.17706579641132e-5], 80802, 80802)

In [62]:
B1 = A ⋅ Grad(Pu)
B2 = G ⋅ Pu

K2 =
    L.∫(Grad(Pu) ⋅ A' ⋅ M ⋅ A ⋅ Grad(Pu); Ω="body", gauss=:full) +
    L.∫(Grad(Pu) ⋅ A' ⋅ M ⋅ G ⋅ Pu; Ω="body", gauss=:full) +
    L.∫(Pu ⋅ G' ⋅ M ⋅ A ⋅ Grad(Pu); Ω="body", gauss=:full) +
    L.∫(Pu ⋅ G' ⋅ M ⋅ G ⋅ Pu; Ω="body", gauss=:full)

sparse([1, 2, 9, 10, 1599, 1600, 1601, 1602, 1, 2  …  807, 808, 80401, 80402, 80403, 80404, 80799, 80800, 80801, 80802], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802], [0.6650027777777777, -0.0008333333333333333, -0.16666527777777818, -0.0004166666666666663, -0.1674986111111109, -0.0008333333333333336, -0.3333326388888887, -0.00041666666666666664, -0.0008333333333333333, 2.77777777777778e-6  …  0.0004166666666666745, 6.944444444444398e-7, -0.00041666666666667065, 6.94444444444436e-7, 2.0057740190981832e-18, 2.7777777777777284e-6, -0.0016666666666666672, 2.7777777777777826e-6, -1.214306433183765e-17, 1.1111111111111149e-5], 80802, 80802)

In [63]:
K[:, :]

80802×80802 SparseArrays.SparseMatrixCSC{Float64, Int64} with 1440438 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [64]:
K2[:, :]

80802×80802 SparseArrays.SparseMatrixCSC{Float64, Int64} with 1440438 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [65]:
K1 = ∫((B1 + B2)' ⋅ M ⋅ (B1 + B2); Ω="body", gauss=:full)

norm(K1.A - K2.A)

0.0

In [66]:
B = A ⋅ Grad(Pu) + G ⋅ Pu

K3 = ∫(B' ⋅ M ⋅ B; Ω="body", gauss=:full)

sparse([1, 2, 9, 10, 1599, 1600, 1601, 1602, 1, 2  …  807, 808, 80401, 80402, 80403, 80404, 80799, 80800, 80801, 80802], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802], [0.6650027777777777, -0.0008333333333333333, -0.16666527777777818, -0.0004166666666666663, -0.1674986111111109, -0.0008333333333333336, -0.3333326388888887, -0.00041666666666666664, -0.0008333333333333333, 2.77777777777778e-6  …  0.0004166666666666745, 6.944444444444398e-7, -0.00041666666666667065, 6.94444444444436e-7, 2.0057740190981832e-18, 2.7777777777777284e-6, -0.0016666666666666672, 2.7777777777777826e-6, -1.214306433183765e-17, 1.1111111111111149e-5], 80802, 80802)

In [67]:
B = full(A ⋅ Grad(Pu)) + reduced(G ⋅ Pu)

K4 = ∫(B' ⋅ M ⋅ B; Ω="body", gauss=:auto)

sparse([1, 2, 9, 10, 1599, 1600, 1601, 1602, 1, 2  …  807, 808, 80401, 80402, 80403, 80404, 80799, 80800, 80801, 80802], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802], [0.6650035440710214, -0.0008333333333333333, -0.16666501119203062, -0.0004166666666666663, -0.16749937740435455, -0.0008333333333333336, -0.3333329054746363, -0.00041666666666666664, -0.0008333333333333333, 3.5440710214361918e-6  …  0.0004166666666666745, 4.278586968845227e-7, -0.00041666666666667065, 4.278586968845099e-7, 2.0057740190981832e-18, 3.310949272896065e-6, -0.0016666666666666672, 2.4480043512778716e-6, -1.214306433183765e-17, 1.17706579641132e-5], 80802, 80802)

In [68]:
norm(K3.A - K4.A)

0.00034903826991733596

In [69]:
r = ScalarField(Pu, "body", (x, y, z)->x + 0.00000001)

elementwise ScalarField
[[1.0e-8; 0.00500001; 0.005000010000000004; 1.0e-8;;], [1.0e-8; 0.005000010000000004; 0.005000010000000004; 1.0e-8;;], [1.0e-8; 0.005000010000000004; 0.005000010000000004; 1.0e-8;;], [1.0e-8; 0.005000010000000004; 0.005000010000000004; 1.0e-8;;], [1.0e-8; 0.005000010000000004; 0.005000010000000004; 1.0e-8;;], [1.0e-8; 0.005000010000000004; 0.005000010000000004; 1.0e-8;;], [1.0e-8; 0.005000010000000004; 0.005000010000000004; 1.0e-8;;], [1.0e-8; 0.005000010000000004; 0.005000010000000004; 1.0e-8;;], [1.0e-8; 0.005000010000000004; 0.005000010000000171; 1.0e-8;;], [1.0e-8; 0.005000010000000171; 0.005000010000000115; 1.0e-8;;]  …  [0.99500001; 1.00000001; 1.00000001; 0.99500001;;], [0.99500001; 1.00000001; 1.00000001; 0.99500001;;], [0.99500001; 1.00000001; 1.00000001; 0.9950000099999999;;], [0.9950000099999999; 1.00000001; 1.00000001; 0.99500001;;], [0.99500001; 1.00000001; 1.00000001; 0.9950000099999999;;], [0.9950000099999999; 1.00000001; 1.00000001; 0.99500001;;]

In [70]:
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]

4×3 Matrix{Int64}:
 1  0  0
 0  0  0
 0  1  0
 0  0  1

In [71]:
A2 = [0 0; 1/r 0; 0 0; 0 0]

4×2 Matrix{Any}:
 0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [72]:
B = A1 ⋅ SymGrad(Pu) + A2 ⋅ Pu

ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 40401, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.SymGradOp()), Any[[1 0 0 0; 0 0 1 0; 0 0 0 1]]), :full), ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 40401, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.IdOp()), Any[Any[0 ScalarField([[1.0e8; 199.9996000008; 199.99960000079983; 1.0e8;;], [1.0e8; 199.99960000079983; 199.99960000079983; 1.0e8;;], [1.0e8; 199.99960000079983; 199.999

In [73]:
E = mat.E
ν = mat.ν

D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]
R = 2π * [r 0 0 0; 0 r 0 0; 0 0 r 0; 0 0 0 r]

4×4 Matrix{Any}:
  ScalarField([[6.28319e-8; 0.031416; 0.031416; 6.28319e-8;;], [6.28319e-8; 0.031416; 0.031416; 6.28319e-8;;], [6.28319e-8; 0.031416; 0.031416; 6.28319e-8;;], [6.28319e-8; 0.031416; 0.031416; 6.28319e-8;;], [6.28319e-8; 0.031416; 0.031416; 6.28319e-8;;], [6.28319e-8; 0.031416; 0.031416; 6.28319e-8;;], [6.28319e-8; 0.031416; 0.031416; 6.28319e-8;;], [6.28319e-8; 0.031416; 0.031416; 6.28319e-8;;], [6.28319e-8; 0.031416; 0.031416; 6.28319e-8;;], [6.28319e-8; 0.031416; 0.031416; 6.28319e-8;;]  …  [6.25177; 6.28319; 6.28319; 6.25177;;], [6.25177; 6.28319; 6.28319; 6.25177;;], [6.25177; 6.28319; 6.28319; 6.25177;;], [6.25177; 6.28319; 6.28319; 6.25177;;], [6.25177; 6.28319; 6.28319; 6.25177;;], [6.25177; 6.28319; 6.28319; 6.25177;;], [6.25177; 6.28319; 6.28319; 6.25177;;], [6.25177; 6.28319; 6.28319; 6.25177;;], [6.25177; 6.28319; 6.28319; 6.25177;;], [6.25177; 6.28319; 6.28319; 6.25177;;]], Matrix{Float64}(undef, 0, 0), [0.0], [805, 806, 807, 808, 809, 810, 811, 812, 813, 8

In [74]:
K = ∫(B' ⋅ D ⋅ B, weight=2π*r)
K[:, :]

80802×80802 SparseArrays.SparseMatrixCSC{Float64, Int64} with 1444096 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [75]:
R2 = 2π * [r 0; 0 r]

f = L.∫(Pu ⋅ R2 ⋅ [1, 0], Γ="right")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [76]:
bc = BoundaryCondition("leftbottom", uy=0)

BoundaryCondition("leftbottom", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0))

In [77]:
u = solveField(K, f, support=[bc])

nodal VectorField
[-9.854606222512269e-15; 0.0; … ; 3.4822129515054334e-6; -2.952598716888837e-6;;]

In [78]:
showDoFResults(u, name="u", visible=true, factor=1e5)

0

In [79]:
prob = Problem([mat], type=:AxiSymmetric)

ld = load("right", fx=1)

LoadCondition("right", nothing, Dict{Symbol, Union{Nothing, Function, Number, ScalarField}}(:fy => nothing, :fz => nothing, :fx => 1))

In [80]:
K2 = stiffnessMatrix(prob)

sparse([1, 2, 9, 10, 1599, 1600, 1601, 1602, 1, 2  …  807, 808, 80401, 80402, 80403, 80404, 80799, 80800, 80801, 80802], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802], [4631.835322600338, -100.69207223044245, 201.38414446088407, 100.69207223044245, 2013.8414446088434, 503.4603611522104, -201.38414446088387, -503.46036115221034, -100.69207223044245, 1107.6127945348628  …  299558.9148855707, -359772.77407939127, -299558.91488556855, -359772.77407938195, 4.4165062718093395e-9, 239848.51605289243, 1006.9207223112026, -961810.6739451898, -5.471520125865936e-9, 2.8854320218355744e6], 80802, 80802)

In [81]:
f2 = loadVector(prob, [ld])
u2 = solveDisplacement(K2, f2, support=[bc])

nodal VectorField
[2.428058582529261e-18; 0.0; … ; 3.4824999999998307e-6; -2.984999999851308e-6;;]

In [82]:
showDoFResults(u2, name="u2", factor=1e5)

1

In [83]:
D

4×4 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0
 0.0        0.0        0.0        76923.1

In [84]:
A2

4×2 Matrix{Any}:
 0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [85]:
D * A2

4×2 Matrix{Any}:
 ScalarField([[1.15385e13; 2.30769e7; 2.30769e7; 1.15385e13;;], [1.15385e13; 2.30769e7; 2.30769e7; 1.15385e13;;], [1.15385e13; 2.30769e7; 2.30769e7; 1.15385e13;;], [1.15385e13; 2.30769e7; 2.30769e7; 1.15385e13;;], [1.15385e13; 2.30769e7; 2.30769e7; 1.15385e13;;], [1.15385e13; 2.30769e7; 2.30769e7; 1.15385e13;;], [1.15385e13; 2.30769e7; 2.30769e7; 1.15385e13;;], [1.15385e13; 2.30769e7; 2.30769e7; 1.15385e13;;], [1.15385e13; 2.30769e7; 2.30769e7; 1.15385e13;;], [1.15385e13; 2.30769e7; 2.30769e7; 1.15385e13;;]  …  [1.15964e5; 1.15385e5; 1.15385e5; 1.15964e5;;], [1.15964e5; 1.15385e5; 1.15385e5; 1.15964e5;;], [1.15964e5; 1.15385e5; 1.15385e5; 1.15964e5;;], [1.15964e5; 1.15385e5; 1.15385e5; 1.15964e5;;], [1.15964e5; 1.15385e5; 1.15385e5; 1.15964e5;;], [1.15964e5; 1.15385e5; 1.15385e5; 1.15964e5;;], [1.15964e5; 1.15385e5; 1.15385e5; 1.15964e5;;], [1.15964e5; 1.15385e5; 1.15385e5; 1.15964e5;;], [1.15964e5; 1.15385e5; 1.15385e5; 1.15964e5;;], [1.15964e5; 1.15385e5; 1.15385e5; 

In [86]:
openPostProcessor()